# Livrable modélisation


---
## Identification du problème

```
- Définir clairement le périmètre de l’étude
- Reformuler le besoin
- Rappeler les objectifs de l’étude
- Relier votre problème au problème de tournée de véhicules ou autre // A COMPLETER (Evann)
```

---

## Définition mathématique du problème

### Définition Formelle du problème

Le problème mTSP peut-être modélisé à l'aide de la théorie des graphes. Nous considérons un graphe non orienté de réseau routier $G = (V, A)$.

#### Les ensembles

- $V=\{0,1,...,n\}$ : L'ensemble des sommets où 0 est le dépôt (point de départ) et $V' = V \backslash \{0\}$
- $A=\{(i,j):i,j \in V,i \neq j\}$ : L'ensemble des arcs représentant les routes possibles entre les villes.

#### Les paramètres

- $c_{ij}$ : le coût (la durée ou la distance) associé au passage sur l'arc $(i,j)$
    - Gestion de la restriction de passage : Si une route est interdite (travaux, sens interdit), on pose $c_{ij} = \infty$. Si elle est simplement plus coûteuse, on ajuste la valeur de $c_{ij}$ en conséquence.
- $m$ : Le nombre de véhicules utilisés pour couvrir l'ensemble des villes.
#### Les Variables de Décision
- $x_{ij}$ : Variable binaire égale à 1 si un véhicule emprunte l'arc $(i, j)$, 0 sinon.
- $u_i$ : Variable auxiliaire pour l'élimination des sous-tours (représentant l'ordre de passage ou la charge cumulative au sommet $i$).

---

### Formulation Mathématique

L'objectif est de trouver un ensemble de $m$ tournées minimisant le coût global, tout en s'assurant que chaque ville (hors dépôt) est visitée exactement une fois.

#### La fonction objectif

La fonction objectif vise à minimiser la somme totale des coûts (ou durées) de tous les déplacements effectués par la flotte de véhicules :

$$\text{Minimiser } Z = \sum_{i \in V} \sum_{j \in V, j \neq i} c_{ij} x_{ij}$$

#### Les contraintes

1. Visite unique des clients : Chaque ville $j$ (autre que le dépôt) doit être visitée et quittée exactement une fois :

$$\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V'$$

$$\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V'$$

2. Flux au dépôt : Exactement $m$ véhicules doivent quitter le dépôt et $m$ véhicules doivent y revenir :

$$\sum_{j \in V'} x_{0j} = m$$

$$\sum_{i \in V'} x_{i0} = m$$

3. Élimination des sous-tours (Contraintes MTZ) : Pour éviter la formation de boucles fermées qui ne passent pas par le dépôt, on utilise les contraintes de Miller-Tucker-Zemlin :

$$u_i - u_j + (n-m+1)x_{ij} \leq n-m \quad \forall (i, j) \in A, i, j \neq 0 \\
\text{Où } 1 \leq u_i \leq n \text{ pour tout } i \in V' \text{.}$$

4. Restriction des arêtes : Si une arête $(i, j)$ est interdite :
$$x_{ij} = 0 \quad \text{pour tout } (i, j) \in A_{interdits}$$
---

---
## Représentation des données

Pour les données, nous avons pris un jeu de donnée sur simplemap.com ([lien](https://simplemaps.com/data/fr-cities)). Les données contiennent des villes ainsi que leur coordonnée géographique (latitude et longitude) ce qui va nous permettre de pouvoir calculer la distance entre deux villes grâce à la formule [d'haversine](https://fr.wikipedia.org/wiki/Formule_de_haversine).

Puisque nous devons générer les données de manière aléatoire, nous avons décidé d'utiliser des données au départ réelles puisque nous utiliserons les données depuis la dataset précédent, mais en sélectionnant les villes de manières aléatoires. En fonctionnant de cette manière, nous avons à la fois des données réelles avec des vraies villes, mais choisis de façon aléatoire.

Dans ce contexte, l'utilisation de graphe est la plus logique pour représenter les données et les traitées. Nous avons donc avec ces données, décidé de représenter :
- Les sommets du graphes par les différentes villes.
- Les arêtes par le trajet entre deux sommets, villes.
- Le poids des arêtes représente le nombre de kilomètres entre deux sommets/villes.

---
## Complexité
```
- Démontrer la complexité du problème de décision (tournée de véhicules) // NON YULIA/THOMAS
- Adapter l’étude de complexité à votre cas (variante simple + extensions) // NON YULIA/THOMAS
- Montrer que le problème est NP-difficile // NON YULIA/THOMAS
```

### Étude théorique de la complexité : Rigueur mathématique et état de l'art

Avant de commencer à coder notre algorithme pour l'ADEME, il est essentiel de prouver mathématiquement à quel point notre problème est difficile. En informatique théorique, cette étape est cruciale : si l'on prouve que le problème est "trop complexe" pour être résolu de manière parfaite, on justifie scientifiquement le fait d'utiliser des algorithmes d'approximation par la suite.

Pour mener cette démonstration, nous allons procéder en quatre étapes.

#### 1. Transformer notre besoin en "Problème de décision"

**Pourquoi faire cela ?** En mathématiques, il est très difficile de manipuler le concept d'optimisation absolue ("trouver le plus court chemin"). La méthode académique standard consiste à transformer notre recherche d'optimum en une question fermée (réponse par OUI ou NON) par rapport à un seuil donné.

**Formalisation mathématique :**
Définissons une *instance* (un cas de test) de notre problème :
- Un réseau sous forme de graphe G = (V, E), où V est l'ensemble des villes et E les routes praticables (les routes bloquées n'existent pas dans E).
- Une flotte de K véhicules de livraison.
- Un budget maximum autorisé B (en kilomètres ou en temps).

**La question de décision :** *Existe-t-il un ensemble de K véhicules réalisant une tournée partant du dépôt, visitant chaque client de V exactement une fois via les routes E, de telle sorte que le coût total de toutes les tournées soit inférieur ou égal au budget B ?*

#### 2. Prouver que notre problème appartient à la "Classe NP"

**Qu'est-ce que cela signifie ?** Un problème appartient à la classe NP (Non-déterministe Polynomial) si, lorsqu'on nous donne une solution "devinée" (un *certificat*), nous pouvons vérifier très rapidement si elle est correcte ou non, même si la trouver nous aurait pris des années.

**Démonstration :** Supposons que l'on nous fournisse une solution candidate (la liste détaillée des villes visitées par chaque véhicule). Un simple algorithme de vérification va effectuer les tests suivants :
1. Parcourir la liste pour s'assurer qu'aucune ville n'a été oubliée et qu'aucune n'a été visitée deux fois.
2. Vérifier que chaque déplacement entre deux villes emprunte bien une route qui existe dans notre ensemble E (respect des routes bloquées).
3. Additionner les distances et vérifier que le total est $\le B$.

Cette vérification nécessite de parcourir les données une seule fois. Le temps de calcul est donc directement proportionnel au nombre de villes n. En mathématiques, on dit que la vérification se fait en temps polynomial O(n).

**Conclusion de l'étape 2 :** Puisque la vérification est rapide, notre problème de décision appartient formellement à la classe NP.

#### 3. Démonstration de la NP-Complétude (La Preuve par Réduction)

**L'objectif :** Il faut maintenant prouver que notre problème fait partie des "plus difficile" problème de cette classe NP. Pour cela, la méthode mathématique absolue est la réduction polynomiale : on prend un problème déjà connu pour être insoluble rapidement, et on démontre qu'il se cache à l'intérieur de notre propre problème.

Nous allons réduire le problème du Cycle Hamiltonien (trouver un chemin passant par tous les points d'un graphe, reconnu NP-complet) vers le problème du Voyageur de Commerce (TSP), qui est la base de notre projet.

**A. La transformation :**

Prenons un graphe G où l'on cherche un Cycle Hamiltonien. Pour le transformer en Voyageur de Commerce, nous allons inventer des routes pour relier toutes les villes entre elles (créer un graphe complet).
- Si la route existait dans le graphe d'origine G, on lui donne une distance (un poids) de 1.
- Si la route a été inventée, on lui donne un poids de 2 (pour la pénaliser).
- On fixe notre budget maximum B pour qu'il soit exactement égal au nombre de villes |V|.

**B. La preuve par double implication :**
- *Sens 1 :* S'il existe un vrai Cycle Hamiltonien dans le graphe d'origine G, le véhicule pourra faire le tour en n'empruntant que les routes d'origine, donc uniquement des routes de poids 1. Son coût total sera donc exactement |V| $\times 1 = B$. La réponse au problème est OUI.
- *Sens 2 :* S'il existe une tournée de coût $\le B$, alors le véhicule a l'interdiction mathématique d'utiliser une route inventée de poids 2. S'il en utilisait ne serait-ce qu'une seule, le coût minimal monterait à 2 + (|V|-1) = |V| + 1, ce qui dépasse le budget. Il n'a donc emprunté que des routes de poids 1 (les routes d'origine). Cela prouve qu'un Cycle Hamiltonien existe.

Cette démonstration prouve formellement que le Voyageur de Commerce (TSP) est NP-complet.

**C. L'appui de la littérature scientifique :**

Cette démarche mathématique s'appuie sur un socle historique :
- L'article fondateur de **Richard Karp** ("Reducibility Among Combinatorial Problems, 1972") pose les bases de cette démonstration.
- L'ouvrage canonique de **Michael Garey et David Johnson** ("Computers and Intractability, 1979") confirme que le problème des Tournées de Véhicules (VRP) est une généralisation directe du TSP.

**D. Adaptation à notre cas d'étude :**

Notre problème pour l'ADEME est beaucoup plus complexe que le TSP de base : nous avons plusieurs véhicules (K) et un graphe incomplet (routes interdites). Puisque notre modèle englobe le TSP comme un sous-cas extrêmement simplifié (le cas où K=1 et toutes les routes sont ouvertes), il est logiquement au moins aussi difficile.

**Conclusion de l'étape 3 :** La version décisionnelle de notre problème est bien **NP-complète**.

#### 4. Conclusion finale : Le problème d'Optimisation est NP-Difficile

Pour l'ADEME, notre but final n'est pas de répondre "OUI" ou "NON" à un budget, mais de trouver mathématiquement la tournée la plus courte possible (optimisation absolue).
Chercher l'optimum d'un problème NP-complet nous fait basculer dans la classe des problèmes NP-Difficiles.

**Quelles sont les conséquences concrètes pour notre code Python ?**

Le postulat fondamental de l'informatique théorique (P \neq NP) implique qu'il n'existe à ce jour aucun algorithme au monde capable de trouver la solution parfaite à notre problème en un temps de calcul raisonnable (polynomial). Si nous essayions d'utiliser une méthode exacte (comme la force brute, qui évalue toutes les combinaisons de routes), l'espace des solutions augmenterait de manière factorielle O(n!). Pour un réseau comme celui de la France, l'ordinateur mettrait des milliards d'années à terminer son calcul (c'est ce qu'on appelle l'explosion combinatoire).

Cette barrière mathématique indépassable justifie la stratégie que nous adopterons dans les prochaines boucles du projet : nous n'utiliserons pas de méthodes de résolution exactes. Nous allons coder des heuristiques et méta-heuristiques. Ces algorithmes font le compromis d'abandonner la quête de la solution "parfaite à 100%" au profit d'excellentes solutions très proches de l'optimum, calculées en seulement quelques secondes, ce qui répondra parfaitement au besoin opérationnel de l'ADEME.

---
## Contraintes
```
- Expliciter les contraintes // A COMPLETER ARTHUR
```

### Choix des contraintes à ajouter au projet
Les contraintes que nous avons choisis de rajouter au projets sont:
- Coût ou restriction de passage sur certaines arêtes : Certaines routes peuvent être plus coûteuses ou interdites (par exemple, travaux ou routes bloquées).
- Utilisation de plusieurs véhicules : Il peut y avoir plusieurs sous-tournées plutôt qu'une seule grande.

### Explication de ces contraintes

 Dans le problème classique du Voyageur de Commerce, l'objectif algorithmique est de trouver un unique cycle hamiltonien optimal. C'est-à-dire une seule et unique boucle fermée qui visite chaque sommet du graphe exactement une fois, avant de revenir à son point de départ. En introduisant une flotte de K véhicules, la nature même de la solution recherchée dans le graphe change :
 - Génération de multiples cycles : La solution n'est plus un cycle hamiltonien unique, mais un ensemble de N cycles distincts.
  - Le sommet central : Tous ces cycles partagent obligatoirement un et un seul sommet en commun : le nœud d'origine (le dépôt). C'est le point d'intersection de toutes les tournées.
- Sommets disjoints : À l'exception du sommet d'origine, tous les autres sommets du graphe doivent être répartis entre ces les différents cycles. Un sommet appartenant au cycle du véhicule A ne peut pas appartenir au cycle du véhicule B. L'algorithme doit donc non seulement optimiser l'ordre de passage, mais aussi faire un choix de partitionnement des sommets.


Dans un réseau routier réel, des aléas surviennent quotidiennement (travaux, routes barrées, embouteillages importants). Dans notre modélisation, ces aléas ne modifient pas les sommets (les villes restent à leur place), mais viennent directement altérer les arêtes du graphe de deux manières concrètes :
 - Les restrictions totales : Concrètement, cela se traduit par la suppression de X arêtes dans notre graphe. Les algorithmes de résolution ne pourront tout simplement plus emprunter ces chemins. Nous allons supprimer de manières aléatoires des arêtes tout en vérifiant de garder un graphe connexe.
- Les pénalités de coût : Dans ce cas, l'arête existe toujours, mais son "poids" (qui représente le temps de trajet ou le coût) est artificiellement gonflé. Nous allons cibler X  arêtes dont nous allons multiplier le poids par un coefficient de pénalité. Le graphe reste le même, mais les "plus courts chemins" varieront, forçant l'algorithme à réévaluer si un long détour géométrique n'est pas finalement devenu "plus court" en temps qu'une ligne droite embouteillée.


---
FISA A3 CESI 2025-2026
- Hugo  Denombret
- Thomas Courtot
- Arthur Roux
- Yulia Pavlova
- Evann Abrial
